# autoclass-lite — Library Demo

This notebook demonstrates the `autoclass_lite` library on two real-world classification datasets:

| # | Dataset | Classes | Features |
|---|---------|---------|----------|
| 1 | **Iris** (sklearn built-in) | 3 | 4 numerical |
| 2 | **Palmer Penguins** (CSV) | 3 | 4 numerical |

For each dataset we run:
- `SimpleAutoML` — trains all 4 models with K-fold CV and ranks them
- `GridAutoML` — exhaustive hyperparameter search with memoization cache

In [29]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

from autoclass_lite import SimpleAutoML, GridAutoML
from autoclass_lite.automl.observers import ConsoleObserver

print("autoclass_lite imported successfully ✓")

autoclass_lite imported successfully ✓


---
## Part 1 — Iris Dataset

### 1.1 Load Data

In [30]:
iris = load_iris(as_frame=True)
X_iris = iris.data.values
y_iris = iris.target.values

print(f"Shape  : {X_iris.shape}")
print(f"Classes: {iris.target_names.tolist()}")
print(f"Label counts: {dict(zip(*np.unique(y_iris, return_counts=True)))}")
iris.frame.head()

Shape  : (150, 4)
Classes: ['setosa', 'versicolor', 'virginica']
Label counts: {np.int64(0): np.int64(50), np.int64(1): np.int64(50), np.int64(2): np.int64(50)}


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


### 1.2 SimpleAutoML on Iris

In [31]:
print("=" * 55)
print("  SimpleAutoML — Iris")
print("=" * 55)

simple_iris = SimpleAutoML(cv_splits=5, metric="accuracy", random_state=42)
simple_iris.add_observer(ConsoleObserver())
simple_iris.fit(X_iris, y_iris)

print()
simple_iris.summary()

  SimpleAutoML — Iris
Starting AutoML...
  [knn] {'accuracy': np.float64(0.9666666666666666), 'precision': np.float64(0.9659035409035409), 'recall': np.float64(0.9675925925925926), 'f1_score': np.float64(0.9656884974596135)}
  [logistic_regression] {'accuracy': np.float64(0.8066666666666666), 'precision': np.float64(0.8890849673202614), 'recall': np.float64(0.8325925925925924), 'f1_score': np.float64(0.7965380796708765)}
  [naive_bayes] {'accuracy': np.float64(0.9466666666666667), 'precision': np.float64(0.9512205387205386), 'recall': np.float64(0.9409764309764309), 'f1_score': np.float64(0.9443221050019168)}
  [decision_tree] {'accuracy': np.float64(0.9533333333333334), 'precision': np.float64(0.9549371924371923), 'recall': np.float64(0.9432491582491582), 'f1_score': np.float64(0.9478625759576523)}
Done. Best model: knn

model                 accuracy  precision  recall    f1_score  
---------------------------------------------------------------
knn                   0.9667    0.9659

In [32]:
preds_iris = simple_iris.predict(X_iris)
correct = np.sum(preds_iris == y_iris)
print(f"Best model predictions — {correct}/{len(y_iris)} correct ({correct/len(y_iris)*100:.1f}%)")

Best model predictions — 144/150 correct (96.0%)


### 1.3 GridAutoML on Iris

In [33]:
print("=" * 55)
print("  GridAutoML — Iris")
print("=" * 55)

grid_iris = GridAutoML(
    param_grid={
        "knn": [{"k": 1}, {"k": 3}, {"k": 5}, {"k": 7}],
        "logistic_regression": [
            {"learning_rate": 0.01},
            {"learning_rate": 0.1},
            {"learning_rate": 0.5},
        ],
        "naive_bayes": [{}],
        "decision_tree": [{"max_depth": 3}, {"max_depth": 5}, {"max_depth": 7}],
    },
    cv_splits=5,
    metric="accuracy",
    random_state=42,
)
grid_iris.add_observer(ConsoleObserver())
grid_iris.fit(X_iris, y_iris)

print()
grid_iris.summary()

  GridAutoML — Iris
Starting AutoML...
  [knn] {'accuracy': np.float64(0.9733333333333334), 'precision': np.float64(0.9746642246642248), 'recall': np.float64(0.9731481481481481), 'f1_score': np.float64(0.9724662752373913)}
  [logistic_regression] {'accuracy': np.float64(0.9533333333333334), 'precision': np.float64(0.9554473304473305), 'recall': np.float64(0.9524579124579123), 'f1_score': np.float64(0.9513874176895607)}
  [naive_bayes] {'accuracy': np.float64(0.9466666666666667), 'precision': np.float64(0.9512205387205386), 'recall': np.float64(0.9409764309764309), 'f1_score': np.float64(0.9443221050019168)}
  [decision_tree] {'accuracy': np.float64(0.96), 'precision': np.float64(0.9673717948717948), 'recall': np.float64(0.9498148148148149), 'f1_score': np.float64(0.9566762468343196)}
Done. Best model: knn

model                 accuracy  precision  recall    f1_score  
---------------------------------------------------------------
knn                   0.9733    0.9747     0.9731    0

In [34]:
print(f"Grid cache entries used: {len(grid_iris._cache)}  (memoization — no config evaluated twice)")
best_iris = grid_iris.leaderboard()[0]
print(f"Best model : {best_iris['model']}")
print(f"Best params: {best_iris['params']}")
print(f"Accuracy   : {best_iris['accuracy']:.4f}")

Grid cache entries used: 11  (memoization — no config evaluated twice)
Best model : knn
Best params: {'k': 5}
Accuracy   : 0.9733


---
## Part 2 — Palmer Penguins Dataset (CSV)

### 2.1 Load & Explore

In [35]:
df = pd.read_csv("palmerpenguins_original.csv")
print(f"Shape  : {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Shape  : (344, 8)
Columns: ['species', 'island', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex', 'year']


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


In [36]:
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Class distribution:")
print(df["species"].value_counts())

Missing values per column:
species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
year                  0
dtype: int64

Class distribution:
species
Adelie       152
Gentoo       124
Chinstrap     68
Name: count, dtype: int64


### 2.2 Preprocessing

Steps:
1. Keep only the 4 numerical measurement features (drop `island`, `sex`, `year`)
2. Drop the 2 rows that have missing values in the numerical features
3. Encode the `species` string label → integer (0 / 1 / 2)

In [37]:
# 1. Select numerical features
feature_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
df_clean = df[feature_cols + ["species"]].dropna().reset_index(drop=True)

# 2. Encode target
species_map = {s: i for i, s in enumerate(sorted(df_clean["species"].unique()))}
print("Label encoding:", species_map)

X_penguins = df_clean[feature_cols].values.astype(float)
y_penguins = df_clean["species"].map(species_map).values

print(f"\nFinal shape : {X_penguins.shape}")
print(f"Label counts: {dict(zip(*np.unique(y_penguins, return_counts=True)))}")

Label encoding: {'Adelie': 0, 'Chinstrap': 1, 'Gentoo': 2}

Final shape : (342, 4)
Label counts: {np.int64(0): np.int64(151), np.int64(1): np.int64(68), np.int64(2): np.int64(123)}


### 2.3 SimpleAutoML on Penguins

In [38]:
print("=" * 55)
print("  SimpleAutoML — Palmer Penguins")
print("=" * 55)

simple_penguins = SimpleAutoML(cv_splits=5, metric="accuracy", random_state=42)
simple_penguins.add_observer(ConsoleObserver())
simple_penguins.fit(X_penguins, y_penguins)

print()
simple_penguins.summary()

  SimpleAutoML — Palmer Penguins
Starting AutoML...
  [knn] {'accuracy': np.float64(0.7836743393009378), 'precision': np.float64(0.7514431902349032), 'recall': np.float64(0.7208276112577188), 'f1_score': np.float64(0.7193968636489314)}
  [logistic_regression] {'accuracy': np.float64(0.32689684569479965), 'precision': np.float64(0.10896561523159991), 'recall': np.float64(0.3333333333333333), 'f1_score': np.float64(0.16146452772202163)}
  [naive_bayes] {'accuracy': np.float64(0.9677323103154306), 'precision': np.float64(0.9652664982076746), 'recall': np.float64(0.9594510582010581), 'f1_score': np.float64(0.960917938422716)}
  [decision_tree] {'accuracy': np.float64(0.9678175618073317), 'precision': np.float64(0.9597514570318608), 'recall': np.float64(0.966231684981685), 'f1_score': np.float64(0.961834089121884)}
Done. Best model: decision_tree

model                 accuracy  precision  recall    f1_score  
---------------------------------------------------------------
decision_tree    

In [39]:
preds_penguins = simple_penguins.predict(X_penguins)
correct = np.sum(preds_penguins == y_penguins)
print(f"Best model predictions — {correct}/{len(y_penguins)} correct ({correct/len(y_penguins)*100:.1f}%)")

Best model predictions — 341/342 correct (99.7%)


### 2.4 GridAutoML on Penguins

In [40]:
print("=" * 55)
print("  GridAutoML — Palmer Penguins")
print("=" * 55)

grid_penguins = GridAutoML(
    param_grid={
        "knn": [{"k": 1}, {"k": 3}, {"k": 5}, {"k": 7}],
        "logistic_regression": [
            {"learning_rate": 0.01},
            {"learning_rate": 0.1},
            {"learning_rate": 0.5},
        ],
        "naive_bayes": [{}],
        "decision_tree": [{"max_depth": 3}, {"max_depth": 5}, {"max_depth": 7}],
    },
    cv_splits=5,
    metric="accuracy",
    random_state=42,
)
grid_penguins.add_observer(ConsoleObserver())
grid_penguins.fit(X_penguins, y_penguins)

print()
grid_penguins.summary()

  GridAutoML — Palmer Penguins
Starting AutoML...
  [knn] {'accuracy': np.float64(0.8568201193520887), 'precision': np.float64(0.8412804076669931), 'recall': np.float64(0.8199532879505999), 'f1_score': np.float64(0.8233312685824357)}
  [logistic_regression] {'accuracy': np.float64(0.4242114236999147), 'precision': np.float64(0.14177323103154307), 'recall': np.float64(0.3333333333333333), 'f1_score': np.float64(0.1979825541402785)}
  [naive_bayes] {'accuracy': np.float64(0.9677323103154306), 'precision': np.float64(0.9652664982076746), 'recall': np.float64(0.9594510582010581), 'f1_score': np.float64(0.960917938422716)}
  [decision_tree] {'accuracy': np.float64(0.9678175618073317), 'precision': np.float64(0.9597514570318608), 'recall': np.float64(0.966231684981685), 'f1_score': np.float64(0.961834089121884)}
Done. Best model: decision_tree

model                 accuracy  precision  recall    f1_score  
---------------------------------------------------------------
decision_tree        

In [41]:
print(f"Grid cache entries used: {len(grid_penguins._cache)}  (memoization — no config evaluated twice)")
best_penguins = grid_penguins.leaderboard()[0]
print(f"Best model : {best_penguins['model']}")
print(f"Best params: {best_penguins['params']}")
print(f"Accuracy   : {best_penguins['accuracy']:.4f}")

Grid cache entries used: 11  (memoization — no config evaluated twice)
Best model : decision_tree
Best params: {'max_depth': 5}
Accuracy   : 0.9678


---
## Summary

In [42]:
print("╔══════════════════════════════════════════════════════════════╗")
print("║                    RESULTS SUMMARY                          ║")
print("╠══════════════════════╦═══════════════════╦══════════════════╣")
print("║ Dataset              ║ SimpleAutoML Best ║ GridAutoML Best  ║")
print("╠══════════════════════╬═══════════════════╬══════════════════╣")

s_iris   = simple_iris.leaderboard()[0]
g_iris   = grid_iris.leaderboard()[0]
s_peng   = simple_penguins.leaderboard()[0]
g_peng   = grid_penguins.leaderboard()[0]

print(f"║ Iris                 ║ {s_iris['model']:<10} {s_iris['accuracy']:.4f}  ║ {g_iris['model']:<10} {g_iris['accuracy']:.4f} ║")
print(f"║ Palmer Penguins      ║ {s_peng['model']:<10} {s_peng['accuracy']:.4f}  ║ {g_peng['model']:<10} {g_peng['accuracy']:.4f} ║")
print("╚══════════════════════╩═══════════════════╩══════════════════╝")

╔══════════════════════════════════════════════════════════════╗
║                    RESULTS SUMMARY                          ║
╠══════════════════════╦═══════════════════╦══════════════════╣
║ Dataset              ║ SimpleAutoML Best ║ GridAutoML Best  ║
╠══════════════════════╬═══════════════════╬══════════════════╣
║ Iris                 ║ knn        0.9667  ║ knn        0.9733 ║
║ Palmer Penguins      ║ decision_tree 0.9678  ║ decision_tree 0.9678 ║
╚══════════════════════╩═══════════════════╩══════════════════╝
